# Clonamos el repositorio con los modelos y herramientas

In [1]:
!git clone https://github.com/dannasalazar11/Msc_thesis.git

Cloning into 'Msc_thesis'...
remote: Enumerating objects: 583, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 583 (delta 77), reused 0 (delta 0), pack-reused 427 (from 1)
Receiving objects: 100% (583/583), 53.73 MiB | 41.87 MiB/s, done.
Resolving deltas: 100% (354/354), done.


In [2]:
import sys
sys.path.append('/kaggle/working/Msc_thesis')

from TGARNet.utils import get_segmented_data
#from tensorflow.keras.mixed_precision import set_global_policy
#set_global_policy('mixed_float16')


import tensorflow as tf
import numpy as np
import random
import os

# Establecer semilla
seed = 42

# Semillas para módulos principales
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

2025-12-10 02:01:20.908524: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765332081.132068      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765332081.203285      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Importar base de datos segmentada (Segmentos de 4 seg con translape del 50%, es decir, de 2 seg)

In [3]:
X, y, sbjs = get_segmented_data()
X.shape, y.shape, len(sbjs)

((8213, 19, 512), (8213, 2), 8213)

# Definimos el modelo y definimos los hiperparámetros

In [4]:
import os

import pickle

with open("/kaggle/input/ieee-tdah-control-database/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [5]:
import numpy as np
from scipy.signal import welch
from scipy.stats import kurtosis, skew, entropy

from sklearn.decomposition import PCA
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Dropout, Flatten, Dense,
    Bidirectional, LSTM, GRU, MultiHeadAttention, LayerNormalization, concatenate
)
from tensorflow.keras.models import Model


# ===============================================================
# Feature Extraction
# ===============================================================

class FeatureExtractor:
    def __init__(self, fs=128):
        self.fs = fs

    def hjorth(self, x):
        dx = np.diff(x)
        var_x = np.var(x)
        var_dx = np.var(dx)
        mobility = np.sqrt(var_dx / var_x)
        complexity = np.sqrt(np.var(np.diff(dx)) / var_dx) / mobility
        return mobility, complexity

    def spectral_features(self, x):
        freqs, psd = welch(x, fs=self.fs, nperseg=256)
        bands = {
            "delta": (0.5, 4),
            "theta": (4, 8),
            "alpha": (8, 13),
            "beta": (13, 30),
            "gamma": (30, 45),
        }
        band_powers = []
        for low, high in bands.values():
            idx = (freqs >= low) & (freqs <= high)
            band_powers.append(np.sum(psd[idx]))
        spec_entropy = entropy(psd / np.sum(psd))
        return band_powers, spec_entropy

    def extract_features(self, X):
        feat_list = []
        for win in X:
            feats = []
            for ch in range(win.shape[0]):
                x = win[ch]
                feats.append(np.mean(x))
                feats.append(np.std(x))
                feats.append(skew(x))
                feats.append(kurtosis(x))
                hj_mob, hj_comp = self.hjorth(x)
                feats.extend([hj_mob, hj_comp])
                bp, ent = self.spectral_features(x)
                feats.extend(bp)
                feats.append(ent)
            feat_list.append(feats)
        return np.array(feat_list)


# ===============================================================
# Feature Selection Class (PCA + ChiSquare)
# ===============================================================

class FeatureSelector:
    def __init__(self, pca_var=0.95, chi_k=100):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=pca_var)
        self.chi_k = chi_k
        self.chi = None   # se define luego

    def fit_transform(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        X_pca = self.pca.fit_transform(X_scaled)

        # determinar k válido
        n_features = X_pca.shape[1]
        k = min(self.chi_k, n_features)

        if k == n_features:
            # usa todas las features si PCA ya redujo mucho
            self.chi = SelectKBest(chi2, k='all')
        else:
            self.chi = SelectKBest(chi2, k=k)

        X_selected = self.chi.fit_transform(np.abs(X_pca), y)
        return X_selected

    def transform(self, X):
        X_scaled = self.scaler.transform(X)
        X_pca = self.pca.transform(X_scaled)
        return self.chi.transform(np.abs(X_pca))


# ===============================================================
# Integrated DL Model
# ===============================================================

def build_integrated_model(input_shape):
    inp = Input(shape=input_shape)

    # CNN branch
    x1 = Conv1D(64, 3, activation='relu', padding='same')(inp)
    x1 = MaxPooling1D(2)(x1)
    x1 = Dropout(0.5)(x1)
    x1 = Flatten()(x1)
    x1 = Dense(128, activation='relu')(x1)

    # CNN + BiLSTM branch
    x2 = Conv1D(64, 3, activation='relu', padding='same')(inp)
    x2 = MaxPooling1D(2)(x2)
    x2 = Bidirectional(LSTM(64, return_sequences=False))(x2)
    x2 = Dense(128, activation='relu')(x2)

    # GRU + Self Attention branch
    x3 = GRU(64, return_sequences=True)(inp)
    att = MultiHeadAttention(num_heads=2, key_dim=64)(x3, x3)
    x3 = LayerNormalization()(x3 + att)
    x3 = Flatten()(x3)
    x3 = Dense(128, activation='relu')(x3)

    # Fusion
    merged = concatenate([x1, x2, x3])
    x = Dense(256, activation='relu')(merged)
    x = Dropout(0.5)(x)
    out = Dense(2, activation='softmax')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


# --- reutilizamos tus clases anteriores ---
# (FeatureExtractor y FeatureSelector ya los definiste)

def prepare_EEG_pipeline(X, y, fs=128, chi_k=100):
    """
    Aplica TODO el pipeline ANTES del entrenamiento cruzado:
    - Extracción de características
    - Reducción PCA
    - Chi-square selection
    - reshape para DL

    Retorna:
        X_dl   -> listo para entrar a SGKF_CV
        y      -> etiquetas sin modificar
        selector -> objeto para transformar nuevos datos si deseas
    """

    # === 1) Extracción de características ===
    extractor = FeatureExtractor(fs)
    X_feat = extractor.extract_features(X)

    # === 2) Selección de características ===
    selector = FeatureSelector(pca_var=0.95, chi_k=chi_k)
    X_selected = selector.fit_transform(X_feat, y)

    # === 3) reshape en tensor DL ===
    # shape final = (samples, features, 1)
    X_dl = X_selected[:, :, None]

    print("✔ Pipeline EEG completado")
    print("Shape DL:", X_dl.shape)

    return X_dl, y, selector

## Función de entrenamiento

In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score, roc_auc_score, precision_score, recall_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from collections import defaultdict
import tensorflow as tf
import random


def SGKF_CV(model_builder, X, y, sbjs, model_args, compile_args, folds, 
            model_name='', seed=42, epochs=100, batch_size=16):
    
    all_fold_metrics = []
    total_histories = []
    models = {}  

    for fold, (train_subjects, test_subjects) in enumerate(folds):
        print(f"\n{'-'*60}")
        print(f"Fold {fold+1}/{len(folds)}  |  Test subjects: {test_subjects}")
        print(f"{'-'*60}")

        # ============================================================
        # Obtención de índices por sujeto
        # ============================================================
        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        test_idx  = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        sbjs_test = [sbjs[i] for i in test_idx]

        # ============================================================
        # Reinicio en cada fold
        # ============================================================
        tf.keras.backend.clear_session()
        np.random.seed(seed + fold)
        random.seed(seed + fold)
        tf.random.set_seed(seed + fold)

        # ============================================================
        # Construcción del modelo
        # ============================================================
        model = model_builder(**model_args)
        model.compile(**compile_args)

        print("Entrenando modelo...")

        history = model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )

        total_histories.append(history.history)

        # ============================================================
        # Predicciones por fold
        # ============================================================
        preds = model.predict(X_test)
        y_pred = np.argmax(preds, axis=1)

        # Si las etiquetas están one-hot codificadas
        if y_test.ndim > 1:
            y_true = np.argmax(y_test, axis=1)
        else:
            y_true = y_test

        # ============================================================
        # Cálculo de métricas
        # ============================================================
        fold_metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'kappa': cohen_kappa_score(y_true, y_pred),
        }

        # AUC solo si binaria
        if preds.shape[1] == 2:
            fold_metrics['auc'] = roc_auc_score(y_true, preds[:, 1])
        else:
            fold_metrics['auc'] = np.nan

        all_fold_metrics.append(fold_metrics)

        print(f"Fold {fold+1} Metrics: {fold_metrics}")

        # ============================================================
        # Guardar modelo del fold
        # ============================================================
        models[fold] = model

        # ============================================================
        # Accuracy por sujeto
        # ============================================================
        subject_correct = defaultdict(list)

        for yt, yp, sbj in zip(y_true, y_pred, sbjs_test):
            subject_correct[sbj].append(int(yt == yp))

        subject_accuracies = {sbj: np.mean(vals) for sbj, vals in subject_correct.items()}

        print("Accuracy promedio por sujeto:")
        for sbj in test_subjects:
            if sbj in subject_accuracies:
                print(f"  {sbj}: {subject_accuracies[sbj]:.4f}")

    # ============================================================
    # Promedio global
    # ============================================================
    mean_metrics = {}
    for key in all_fold_metrics[0].keys():
        vals = [f[key] for f in all_fold_metrics]
        mean_metrics[f'mean_{key}'] = np.mean(vals)
        mean_metrics[f'std_{key}'] = np.std(vals)

    print("\nResumen general:")
    for k,v in mean_metrics.items():
        print(f"{k}: {v:.4f}")

    return all_fold_metrics, models, total_histories

In [7]:
# === preprocesar una sola vez ===

X_dl, y_proc, selector = prepare_EEG_pipeline(X, y, fs=128, chi_k=100)

✔ Pipeline EEG completado
Shape DL: (8213, 79, 1)


In [8]:
model_args={'input_shape': X_dl.shape[1:]}
model = build_integrated_model(**model_args)

model.summary()

I0000 00:00:1765332373.889803      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1765332373.890429      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 79, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ gru (GRU)                 │ (None, 79, 64)         │         12,864 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d (Conv1D)           │ (None, 79, 64)         │            256 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention      │ (None, 79, 64)         │         33,216 │ gru[0][0], gru[0][0]   │
│ (MultiHeadAttention)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling1d             │ (None, 39, 64)         │              0 │ conv1d[0][0]           │
│ (MaxPooling1D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_1 (Conv1D)         │ (None, 79, 64)         │            256 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 79, 64)         │              0 │ gru[0][0],             │
│                           │                        │                │ multi_head_attention[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 39, 64)         │              0 │ max_pooling1d[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling1d_1           │ (None, 39, 64)         │              0 │ conv1d_1[0][0]         │
│ (MaxPooling1D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization       │ (None, 79, 64)         │            128 │ add[0][0]              │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten (Flatten)         │ (None, 2496)           │              0 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional             │ (None, 128)            │         66,048 │ max_pooling1d_1[0][0]  │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_1 (Flatten)       │ (None, 5056)           │              0 │ layer_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 128)            │        319,616 │ flatten[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 128)            │         16,512 │ bidirectional[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 128)            │        647,296 │ flatten_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concaten

 Total params: 1,195,266 (4.56 MB)

 Trainable params: 1,195,266 (4.56 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
results = {}

for i in range(10):
    result, models, total_histories = SGKF_CV(
                                    model_builder=build_integrated_model,  
                                    X=X_dl,
                                    y=y_proc,
                                    sbjs=sbjs,         # tus etiquetas de sujeto
                                    model_args={'input_shape': X_dl.shape[1:]},   # argumento para el constructor
                                    compile_args={
                                        'optimizer':'adam',
                                        'loss':'categorical_crossentropy',
                                        'metrics':['accuracy']
                                    },
                                    folds=folds,       # tus folds de validación
                                    model_name='EEG_ADHD',
                                )
    results[i]=result


------------------------------------------------------------
Fold 1/5  |  Test subjects: ['v28p', 'v274', 'v1p', 'v231', 'v22p', 'v29p', 'v206', 'v238', 'v31p', 'v35p', 'v177', 'v200', 'v112', 'v113', 'v48p', 'v140', 'v131', 'v125', 'v55p', 'v143', 'v43p', 'v305', 'v134', 'v114']
------------------------------------------------------------
Entrenando modelo...


I0000 00:00:1765332383.803650      73 cuda_dnn.cc:529] Loaded cuDNN version 90300


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Fold 1 Metrics: {'accuracy': 0.632120796156486, 'recall': 0.6277055296267766, 'precision': 0.6332923445579518, 'kappa': 0.25730567031094265, 'auc': 0.6730944235503799}
Accuracy promedio por sujeto:
  v28p: 0.7358
  v274: 0.8333
  v1p: 0.2391
  v231: 0.5263
  v22p: 0.6304
  v29p: 0.6344
  v206: 0.5325
  v238: 0.9730
  v31p: 0.9545
  v35p: 0.9655
  v177: 0.9219
  v200: 0.9792
  v112: 0.7541
  v113: 0.8305
  v48p: 0.2051
  v140: 0.0303
  v131: 0.3125
  v125: 0.3898
  v55p: 0.8704
  v143: 0.2542
  v43p: 0.3958
  v305: 0.8953
  v134: 0.2549
  v114: 0.9800

------------------------------------------------------------
Fold 2/5  |  Test subjects: ['v18p', 'v39p', 'v234', 'v32p', 'v190', 'v6p', 'v254', 'v204', 'v24p', 'v183', 'v246', 'v219', 'v298', 'v41p', 'v47p', 'v308', 'v52p', 'v300', 'v59p', 'v299', 'v302', 'v51p', 'v109', 'v127']
------------------------------------------------------------
Entrenando modelo...
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step

In [10]:
results

{0: [{'accuracy': 0.632120796156486,
   'recall': 0.6277055296267766,
   'precision': 0.6332923445579518,
   'kappa': 0.25730567031094265,
   'auc': 0.6730944235503799},
  {'accuracy': 0.6480815347721822,
   'recall': 0.654211852562206,
   'precision': 0.6528651010124358,
   'kappa': 0.30181037162961966,
   'auc': 0.7042477778085454},
  {'accuracy': 0.6821266968325792,
   'recall': 0.6843004209950427,
   'precision': 0.6772448273782254,
   'kappa': 0.3573345855275839,
   'auc': 0.741021997157783},
  {'accuracy': 0.5981031416716064,
   'recall': 0.5741029988009386,
   'precision': 0.5834148664793826,
   'kappa': 0.1530321789509852,
   'auc': 0.6270280775431288},
  {'accuracy': 0.6436007348438457,
   'recall': 0.6390289848546254,
   'precision': 0.6589098277122201,
   'kappa': 0.2804838533843693,
   'auc': 0.6438375287071645}],
 1: [{'accuracy': 0.667124227865477,
   'recall': 0.6626867249686589,
   'precision': 0.6701136354662687,
   'kappa': 0.32784783276594975,
   'auc': 0.71534448774